# TC-LoRA on CIFAR-100 and cell states

Baseline 1 of 3. Trains a base diffusion model, freezes it, trains a TC-LoRA
hypernetwork on top, and compares samples.

**Paper:** Cho, Ohana, Jacobsen, Jothi, Chen, Mao, Can.
*TC-LoRA: Temporally Modulated Conditional LoRA for Adaptive Diffusion Control.*
NeurIPS 2025 SpaVLE workshop. [arXiv:2510.09561](https://arxiv.org/abs/2510.09561)

**The authors released no code.** Checked 2026-09-28: no link on the arXiv listing, none
on the project page. `src/guidance/baselines/tc_lora.py` is our reimplementation from
Eq. 1 and Appendices A-B; its header lists every deviation. Report this as a
reimplementation, not as their result reproduced.

Set `MODALITY` in the config cell and run top to bottom.

In [ ]:
# Colab bootstrap. Skip if the package is already importable.
import importlib.util, subprocess, sys, os
if importlib.util.find_spec("guidance") is None:
    if not os.path.exists("discrete-generative-models-proj-1"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                       check=True)
    sys.path.insert(0, "discrete-generative-models-proj-1/src")
from guidance.baselines.tc_lora import TCLoRA
print("imported TCLoRA from", TCLoRA.__module__)

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MODALITY = "images"        # "images" or "cells"

@dataclass
class Cfg:
    modality: str = MODALITY
    seed: int = 0
    batch_size: int = 128
    base_epochs: int = 20          # raise for real runs
    lora_epochs: int = 10
    lr_base: float = 2e-4
    lr_hyper: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # DDIM steps
    rank: int = 4
    hyper_width: int = 256
    subset: int = 0                # >0 truncates the dataset, for smoke tests

cfg = Cfg()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

Two things per modality.

**`make_loaders(cfg)`** returns `(train_loader, val_loader, meta)`. Loaders yield
`(x, cond_id)`:

| | `x` | `cond_id` |
|---|---|---|
| images | `(B, 3, 32, 32)` float in `[-1, 1]` | `(B,)` long in `[0, 100)` |
| cells | `(B, D)` float, standardised | `(B,)` long in `[0, n_clusters)` |

**`meta`** is a dict with `n_conditions`, `shape` (one sample, no batch axis), and
`cond_text` — one short description per condition, read by the frozen text encoder.

CIFAR-100 works as-is. **Drop your cell loader into `make_cell_loaders`.**

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, Subset

def make_image_loaders(cfg):
    from torchvision import datasets, transforms
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),   # -> [-1, 1]
    ])
    full = datasets.CIFAR100(root="data", train=True, download=True, transform=tf)
    g = torch.Generator().manual_seed(cfg.seed)
    perm = torch.randperm(len(full), generator=g).tolist()
    if cfg.subset:
        perm = perm[:cfg.subset]
    cut = int(0.9 * len(perm))
    meta = {
        "n_conditions": 100,
        "shape": (3, 32, 32),
        "cond_text": [f"a photo of a {c.replace('_', ' ')}" for c in full.classes],
    }
    return (DataLoader(Subset(full, perm[:cut]), cfg.batch_size, shuffle=True, drop_last=True),
            DataLoader(Subset(full, perm[cut:]), cfg.batch_size, shuffle=False), meta)


def make_cell_loaders(cfg):
    '''REPLACE THIS BODY with your Tahoe-100M loader.

    Return (train_loader, val_loader, meta) matching the contract above:
      x        (B, D) float32, standardised using TRAIN-split statistics only
      cond_id  (B,)   long, cluster or drug index
      meta     {"n_conditions": k, "shape": (D,), "cond_text": [str] * k}

    cond_text should describe the biology, not just name the cluster, e.g.
      "Trametinib, a MEK1/2 inhibitor that blocks MAPK/ERK signalling"
    Write these once from a public source BEFORE looking at any results.

    Until replaced, this generates a synthetic mixture so the notebook runs end to end.
    '''
    D, k, n = 50, 3, 6000
    rng = np.random.default_rng(cfg.seed)
    centers = rng.normal(size=(k, D)) * 3.0
    spreads = np.linspace(0.4, 1.4, k)          # clusters differ in spread on purpose
    cond = rng.integers(0, k, size=n)
    x = centers[cond] + rng.normal(size=(n, D)) * spreads[cond][:, None]
    x = (x - x.mean(0)) / (x.std(0) + 1e-8)

    ds = TensorDataset(torch.tensor(x, dtype=torch.float32),
                       torch.tensor(cond, dtype=torch.long))
    cut = int(0.9 * n)
    meta = {
        "n_conditions": k,
        "shape": (D,),
        "cond_text": [f"synthetic perturbation cluster {i}" for i in range(k)],
        "SYNTHETIC": True,
    }
    return (DataLoader(Subset(ds, range(cut)), cfg.batch_size, shuffle=True, drop_last=True),
            DataLoader(Subset(ds, range(cut, n)), cfg.batch_size, shuffle=False), meta)


train_loader, val_loader, meta = (
    make_image_loaders(cfg) if cfg.modality == "images" else make_cell_loaders(cfg))

if meta.get("SYNTHETIC"):
    print("WARNING: synthetic cell data. Replace make_cell_loaders before reporting anything.")

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), float(xb.min()), float(xb.max()), "| cond", tuple(cb.shape))
print("conditions:", meta["n_conditions"], "| example text:", meta["cond_text"][0])

## Frozen condition encoder

`h(c)` is a frozen text embedding of the condition description. The same encoder serves
both modalities, which is what makes the transfer claim a real one.

Falls back to fixed random vectors if `transformers` is unavailable so the notebook still
runs. **A fallback run is not a result** — it makes the text meaningless.

In [ ]:
def build_cond_embeddings(texts, device):
    try:
        from transformers import CLIPTokenizer, CLIPTextModel
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), 64):
                batch = tok(texts[i:i + 64], padding=True, truncation=True, return_tensors="pt")
                batch = {k: v.to(device) for k, v in batch.items()}
                out.append(enc(**batch).pooler_output)
        emb = torch.cat(out)
        return emb / emb.norm(dim=-1, keepdim=True), False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback embeddings.")
        g = torch.Generator().manual_seed(0)
        emb = torch.randn(len(texts), 512, generator=g)
        return (emb / emb.norm(dim=-1, keepdim=True)).to(device), True

COND_EMB, IS_FALLBACK = build_cond_embeddings(meta["cond_text"], DEV)
COND_DIM = COND_EMB.shape[1]
print("condition embeddings:", tuple(COND_EMB.shape), "| fallback:", IS_FALLBACK)

## Backbones — a U-Net for images, an MLP for cells

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)


def conv_block(cin, cout, groups=8):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(groups, cout), nn.SiLU(),
    )


class UNet(nn.Module):
    '''Class-conditional U-Net. The label enters as a per-channel bias after block 1.'''

    def __init__(self, c=64, n_classes=100, tdim=128):
        super().__init__()
        self.tdim, self.null = tdim, n_classes
        self.t_mlp = nn.Sequential(nn.Linear(tdim, c), nn.SiLU(), nn.Linear(c, c))
        self.label = nn.Embedding(n_classes + 1, c)      # +1 = null token
        self.e1, self.e2 = conv_block(3, c), conv_block(c, 2 * c)
        self.mid = conv_block(2 * c, 4 * c)
        self.d2, self.d1 = conv_block(6 * c, 2 * c), conv_block(3 * c, c)
        self.out = nn.Conv2d(c, 3, 1)
        self.pool = nn.MaxPool2d(2)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        bias = (self.t_mlp(timestep_embedding(t, self.tdim))
                + self.label(cond_ids))[:, :, None, None]
        s1 = self.e1(x) + bias
        s2 = self.e2(self.pool(s1))
        h = self.mid(self.pool(s2))
        h = self.d2(torch.cat([F.interpolate(h, size=s2.shape[-2:]), s2], 1))
        h = self.d1(torch.cat([F.interpolate(h, size=s1.shape[-2:]), s1], 1))
        return self.out(h)


class CellMLP(nn.Module):
    '''Plain MLP. Width follows BranchSBM's flow network (3 x 1024, SELU).'''

    def __init__(self, d=50, n_classes=3, width=1024, tdim=128):
        super().__init__()
        self.tdim, self.null = tdim, n_classes
        self.t_mlp = nn.Sequential(nn.Linear(tdim, width), nn.SiLU(), nn.Linear(width, width))
        self.label = nn.Embedding(n_classes + 1, width)
        self.inp = nn.Linear(d, width)
        self.body = nn.Sequential(nn.SELU(), nn.Linear(width, width),
                                  nn.SELU(), nn.Linear(width, width), nn.SELU())
        self.out = nn.Linear(width, d)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        h = self.inp(x) + self.t_mlp(timestep_embedding(t, self.tdim)) + self.label(cond_ids)
        return self.out(self.body(h))


def make_backbone(cfg, meta):
    if cfg.modality == "images":
        return UNet(64, meta["n_conditions"])
    return CellMLP(meta["shape"][0], meta["n_conditions"])

net = make_backbone(cfg, meta).to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]


def add_noise(x0):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device)
    eps = torch.randn_like(x0)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


# Range to clamp the predicted clean sample to. At the first DDIM step sqrt(abar)
# is ~0.003, so x0 = (x - sqrt(1-abar) eps) / sqrt(abar) divides by a tiny number and
# any error in eps is amplified ~300x. Without this the samples blow up and the
# variance ratio comes out in the thousands. Images live in [-1, 1]; for standardised
# cell data we take the training range.
CLIP_X0 = 1.0 if cfg.modality == "images" else float(
    torch.cat([x for x, _ in train_loader]).abs().max())
print("x0 clamp:", CLIP_X0)


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None):
    '''eps_fn(x, t) -> predicted noise. Deterministic DDIM.'''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    x = torch.randn(n, *shape, device=DEV)
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-clip_x0, clip_x0)
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return x

## Train the base model

In [ ]:
def run_epochs(model, params, n_epochs, lr, tag, loss_fn):
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best = [], float("inf")
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = loss_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        model.eval(); vtot = vnum = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEV), y.to(DEV)
                vtot += loss_fn(x, y).item() * x.shape[0]; vnum += x.shape[0]
        val = vtot / vnum
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": val < best})
        best = min(best, val)
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}  ({time.time()-t0:.0f}s)")
    return hist


def base_loss(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)

base_hist = run_epochs(net, net.parameters(), cfg.base_epochs, cfg.lr_base, "base", base_loss)
torch.save({"model": net.state_dict(), "hist": base_hist, "cfg": asdict(cfg)},
           f"base_{cfg.modality}.pt")

## Attach TC-LoRA and train the hypernetwork

The backbone is frozen; only the hypernetwork trains. The first assertion is the one that
matters — at initialisation the adapted model must match the base model exactly, because
the hypernetwork's B head is zero-initialised (Appendix A). If that fails, the adapter is
not a clean control and the ablation means nothing.

In [ ]:
base_frozen = make_backbone(cfg, meta).to(DEV)
base_frozen.load_state_dict(torch.load(f"base_{cfg.modality}.pt", map_location=DEV)["model"])
base_frozen.eval()

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, probe_y).clone()

tc = TCLoRA(base_frozen, cond_dim=COND_DIM, rank=cfg.rank,
            width=cfg.hyper_width, skip_names=("out",)).to(DEV)

with torch.no_grad():
    got = tc(probe_x, probe_t, COND_EMB[probe_y], cond_ids=probe_y)
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(n.startswith("hyper.") for n, p in tc.named_parameters() if p.requires_grad)
print(f"trainable: {tc.n_trainable():,} params, all in the hypernetwork")
print(f"frozen:    {sum(p.numel() for p in base_frozen.parameters()):,} params")
print(f"adapted layers: {len(tc.adapted)}")

In [ ]:
def tc_loss(x, y):
    xk, k, eps = add_noise(x)
    # The hypernetwork reads h(c); the frozen backbone still gets the label id.
    pred = tc(xk, k.float() / cfg.K, COND_EMB[y], cond_ids=y)
    return F.mse_loss(pred, eps)

tc_hist = run_epochs(tc, [p for p in tc.parameters() if p.requires_grad],
                     cfg.lora_epochs, cfg.lr_hyper, "tc-lora", tc_loss)
torch.save({"hyper": tc.hyper.state_dict(), "hist": tc_hist, "cfg": asdict(cfg)},
           f"tclora_{cfg.modality}.pt")

## Sample from both

In [ ]:
N = 256 if cfg.modality == "cells" else 16
y_eval = torch.arange(N, device=DEV) % meta["n_conditions"]

base_s = ddim_sample(lambda x, t: base_frozen(x, t, y_eval), N, meta["shape"], cfg.sample_steps)
tc_s = ddim_sample(lambda x, t: tc(x, t, COND_EMB[y_eval], cond_ids=y_eval),
                   N, meta["shape"], cfg.sample_steps)
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps)

In [ ]:
import matplotlib.pyplot as plt

if cfg.modality == "images":
    fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
    for row, (name, s) in enumerate([("base", base_s), ("tc-lora", tc_s)]):
        for j in range(8):
            axes[row, j].imshow(((s[j].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy())
            axes[row, j].set_xticks([]); axes[row, j].set_yticks([])
        axes[row, 0].set_ylabel(name, fontsize=11)
    plt.tight_layout(); plt.show()
else:
    real = torch.cat([x for x, _ in val_loader]).numpy()
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True, sharey=True)
    for a, (name, pts) in zip(ax, [("real (val)", real), ("base", base_s.cpu().numpy()),
                                   ("tc-lora", tc_s.cpu().numpy())]):
        a.scatter(pts[:, 0], pts[:, 1], s=8, alpha=0.5)
        a.set_title(name); a.set_xlabel("PC1")
    ax[0].set_ylabel("PC2"); plt.tight_layout(); plt.show()

## Metrics and manifest

For cell states: RBF-MMD across all dimensions plus the variance ratio. Those are the
columns that sit beside BranchSBM's Table 5, and unlike cluster accuracy they are not
circular. For images this only records validation loss — FID needs 10k+ samples and
belongs in `scripts/evaluate.py`.

In [ ]:
def mmd_rbf(x, y, bandwidth=None, max_n=500, seed=0):
    rng = np.random.default_rng(seed)
    if len(x) > max_n: x = x[rng.choice(len(x), max_n, replace=False)]
    if len(y) > max_n: y = y[rng.choice(len(y), max_n, replace=False)]
    z = np.vstack([x, y])
    d2 = ((z[:, None] - z[None]) ** 2).sum(-1)
    if bandwidth is None:
        med = np.median(d2[d2 > 0])
        bandwidth = float(np.sqrt(med / 2)) if med > 0 else 1.0
    k = np.exp(-d2 / (2 * bandwidth ** 2))
    n, m = len(x), len(y)
    kxx = (k[:n, :n].sum() - np.trace(k[:n, :n])) / max(n * (n - 1), 1)
    kyy = (k[n:, n:].sum() - np.trace(k[n:, n:])) / max(m * (m - 1), 1)
    return float(kxx + kyy - 2 * k[:n, n:].mean()), bandwidth


def variance_ratio(pred, real):
    return float(np.trace(np.cov(pred, rowvar=False)) / np.trace(np.cov(real, rowvar=False)))


report = {
    "method": "TC-LoRA (reimplementation of arXiv:2510.09561)",
    "modality": cfg.modality,
    "config": asdict(cfg),
    "base_val_loss": base_hist[-1]["val"],
    "tclora_val_loss": tc_hist[-1]["val"],
    "trainable_hypernetwork": tc.n_trainable(),
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "synthetic_data": bool(meta.get("SYNTHETIC")),
}

if cfg.modality == "cells":
    real = torch.cat([x for x, _ in val_loader]).numpy()
    for name, s in [("base", base_s), ("tclora", tc_s)]:
        arr = s.cpu().numpy()
        m, bw = mmd_rbf(arr, real)
        report[f"{name}_mmd"] = m
        report[f"{name}_mmd_bandwidth"] = bw
        report[f"{name}_var_ratio"] = variance_ratio(arr, real)

print(json.dumps(report, indent=2))
with open(f"manifest_tclora_{cfg.modality}.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `make_cell_loaders` replaced with the real Tahoe-100M loader
- [ ] `cond_text` written from a public source **before** looking at results
- [ ] CLIP actually loaded — `clip_fallback: false` in the manifest
- [ ] `base_epochs` and `lora_epochs` raised past smoke-test values
- [ ] Three seeds, mean ± std
- [ ] Parameter counts and NFE next to every number
- [ ] Described as a **reimplementation** of arXiv:2510.09561, not a reproduction, with
      the scale gap disclosed: the authors train 251M hypernetwork parameters for 3 days
      on 8× H100.